In [1]:
# %% [markdown]
# Q2: Image Colourization using CNNs (CIFAR-10)
# Setup: imports, reproducibility, device

# %%
import os, math, random, json, time, pathlib
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms, utils as vutils
import matplotlib.pyplot as plt

import wandb

# -----------------
# Reproducibility
# -----------------
# fixed from roll 2023102069
SEED = 2023102069
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# -----------------
# Paths & constants
# -----------------
PROJECT = "smai-q2-colorization"
ENTITY  = "garimamittal643"            # your W&B username
DATA_DIR = "./data_q2"
CENTROIDS_PATH = "./color_centroids.npy"  # provided file (shape [24,3])

# Model dims
NIC = 1        # input channels (grayscale)
NC  = 24       # number of colour classes
IMG_SIZE = 32

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cuda')

In [2]:
# %% [markdown]
# W&B login (Option A: just run wandb.login() in this cell)

# %%
# If you're already logged in, this is a no-op. Otherwise it will prompt a key once.
wandb.login()
print("W&B ready. Entity:", ENTITY, "| Project:", PROJECT)


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: garimamittal643 (garimamittal643-international-institute-of-information-t) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B ready. Entity: garimamittal643 | Project: smai-q2-colorization


In [3]:
# %% [markdown]
# Load 24 RGB colour centroids and define helpers:
#  - rgb_to_gray
#  - encode_to_labels (nearest centroid per pixel)
#  - decode_to_rgb (map label map back to RGB image via centroids)

# %%
def load_centroids(path=CENTROIDS_PATH):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Expected {path} (24x3 RGB cluster centroids). "
            "Place the provided 'color_centroids.npy' next to this notebook."
        )
    C = np.load(path).astype(np.float32)   # [24, 3], values in [0,255] or [0,1]
    # Normalize to [0,1] for consistency
    if C.max() > 1.5:
        C = C / 255.0
    return torch.from_numpy(C)  # [24,3], float32

CENTROIDS = load_centroids().to(device)

def rgb_to_gray(rgb_np):
    """rgb_np: (H,W,3) in [0,1] -> (H,W,1) grayscale in [0,1]."""
    r, g, b = rgb_np[...,0], rgb_np[...,1], rgb_np[...,2]
    gray = 0.2989*r + 0.5870*g + 0.1140*b
    return gray[..., None]

def encode_to_labels(rgb_np, centroids):
    """
    rgb_np: (H,W,3) in [0,1]
    centroids: torch [24,3] on any device
    returns labels (H,W) int64 in [0,23]
    """
    H, W, _ = rgb_np.shape
    px = torch.from_numpy(rgb_np.reshape(-1,3)).float().to(centroids.device)  # [H*W,3]
    # squared Euclidean to all centroids: [H*W,24]
    d2 = torch.cdist(px[None, ...], centroids[None, ...], p=2).squeeze(0)  # [H*W,24]
    labels = d2.argmin(dim=1).view(H, W).to(torch.long).cpu().numpy()
    return labels

def decode_to_rgb(label_map_np, centroids):
    """
    label_map_np: (H,W) ints in [0,23]
    returns rgb_np: (H,W,3) in [0,1]
    """
    C = centroids.detach().cpu().numpy()
    rgb = C[label_map_np.reshape(-1)].reshape(label_map_np.shape + (3,))
    return rgb.clip(0,1)


In [4]:
# %% [markdown]
# Dataset:
# - Input: grayscale [1,32,32]
# - Target: per-pixel class labels [32,32] (0..23) via nearest centroid.
# We'll precompute label maps once for speed.

# %%
class Cifar10Colorize(Dataset):
    def __init__(self, root, train=True, centroids=CENTROIDS, precompute=True):
        self.base = datasets.CIFAR10(root=root, train=train, download=True)
        self.centroids = centroids
        self.precompute = precompute

        # hold grayscale tensor and label map if precomputing
        self.gray_list = []
        self.label_list = []

        if self.precompute:
            for img, _ in self.base:
                # PIL -> np [H,W,3] in [0,1]
                rgb = np.asarray(img).astype(np.float32) / 255.0
                gray = rgb_to_gray(rgb).transpose(2,0,1)           # [1,H,W]
                labels = encode_to_labels(rgb, self.centroids)      # [H,W]
                self.gray_list.append(torch.from_numpy(gray).float())
                self.label_list.append(torch.from_numpy(labels).long())

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        if self.precompute:
            x = self.gray_list[idx]
            y = self.label_list[idx]
        else:
            img, _ = self.base[idx]
            rgb = np.asarray(img).astype(np.float32)/255.0
            gray = rgb_to_gray(rgb).transpose(2,0,1)
            y = encode_to_labels(rgb, self.centroids)
            x = torch.from_numpy(gray).float()
            y = torch.from_numpy(y).long()
        return x, y

# -----------------------
# Build train/val/test
# -----------------------
full_train = Cifar10Colorize(DATA_DIR, train=True, centroids=CENTROIDS, precompute=True)
test_set   = Cifar10Colorize(DATA_DIR, train=False, centroids=CENTROIDS, precompute=True)

# 45k / 5k split
VAL_SIZE = 5000
TRAIN_SIZE = len(full_train) - VAL_SIZE
train_set, val_set = random_split(
    full_train, [TRAIN_SIZE, VAL_SIZE],
    generator=torch.Generator().manual_seed(SEED)
)

def make_loader(ds, batch_size, shuffle, num_workers=2):
    return DataLoader(
        ds, batch_size=batch_size, shuffle=shuffle,
        num_workers=num_workers, pin_memory=(device.type=="cuda")
    )

print(f"Train/Val/Test sizes: {len(train_set)} / {len(val_set)} / {len(test_set)}")


100%|██████████| 170M/170M [00:14<00:00, 12.1MB/s]


Train/Val/Test sizes: 45000 / 5000 / 10000


In [5]:
# %% [markdown]
# Model: Conv blocks down (×3) + ConvTranspose up (×3) + 1×1 classifier.
# Follow the recommended block sequence in the PDF exactly.

# %%
class ConvBNReLU(nn.Module):
    def __init__(self, cin, cout, k):
        super().__init__()
        pad = k // 2  # "same" spatial size for stride=1
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, kernel_size=k, stride=1, padding=pad, bias=False),
            nn.BatchNorm2d(cout),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class DeconvBNReLU(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        # learned upsample: stride=2, kernel=2 -> doubles H,W exactly
        self.net = nn.Sequential(
            nn.ConvTranspose2d(cin, cout, kernel_size=2, stride=2, bias=False),
            nn.BatchNorm2d(cout),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class ColorizationNet(nn.Module):
    def __init__(self, NIC=1, NF=16, NC=24, kernel_size=3):
        super().__init__()
        k = kernel_size

        # Encoder (down)
        self.enc1 = ConvBNReLU(NIC,   NF,   k)  # -> pool -> [NF, 16,16]
        self.enc2 = ConvBNReLU(NF,   2*NF,  k)  # -> pool -> [2NF, 8,8]
        self.enc3 = ConvBNReLU(2*NF, 4*NF,  k)  # -> pool -> [4NF, 4,4] (extra downsample)

        self.pool = nn.MaxPool2d(2)

        # Decoder (up)
        self.dec1 = DeconvBNReLU(4*NF, 2*NF)    # [2NF, 8,8]
        self.dec2 = DeconvBNReLU(2*NF,   NF)    # [NF, 16,16]
        self.dec3 = DeconvBNReLU(NF,      NC)   # [NC, 32,32]

        # 1×1 classifier (kept as in spec)
        self.classifier = nn.Conv2d(NC, NC, kernel_size=1, bias=True)

    def forward(self, x):
        x = self.pool(self.enc1(x))       # [NF,16,16]
        x = self.pool(self.enc2(x))       # [2NF,8,8]
        x = self.pool(self.enc3(x))       # [4NF,4,4]

        x = self.dec1(x)                  # [2NF,8,8]
        x = self.dec2(x)                  # [NF,16,16]
        x = self.dec3(x)                  # [NC,32,32]

        logits = self.classifier(x)       # [NC,32,32]
        return logits

# Quick sanity check
tmp = torch.randn(2, NIC, IMG_SIZE, IMG_SIZE)
model = ColorizationNet(NIC=NIC, NF=16, NC=NC, kernel_size=3)
with torch.no_grad():
    out = model(tmp)
print("Output shape:", tuple(out.shape))


Output shape: (2, 24, 32, 32)


In [6]:
def make_optimizer(model, name, lr, wd=0.0, momentum=0.9):
    if name.lower() == "sgd":
        return torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=wd, nesterov=True)
    elif name.lower() == "adam":
        return torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    else:
        raise ValueError("optimizer must be 'SGD' or 'Adam'")

@torch.no_grad()
def colourize_with_centroids(logits, centroids=CENTROIDS):
    """
    logits: [B, NC, H, W] -> RGB in [0,1], shape [B,3,H,W]
    """
    labels = logits.argmax(dim=1)  # [B,H,W]
    C = centroids.detach().float()              # [NC,3] - Keep on device
    rgb = F.one_hot(labels, num_classes=NC).float()   # [B,H,W,NC]
    rgb = torch.einsum("bhwn,nc->bhwc", rgb, C)       # [B,H,W,3]
    return rgb.permute(0,3,1,2).clamp(0,1)

@torch.no_grad()
def gt_rgb_from_labels(lbls, centroids=CENTROIDS):
    """lbls: [B,H,W] -> [B,3,H,W]"""
    C = centroids.detach().float() # Keep on device
    one = F.one_hot(lbls, num_classes=NC).float()  # [B,H,W,NC] - Keep on device
    rgb = torch.einsum("bhwn,nc->bhwc", one, C)
    return rgb.permute(0,3,1,2).clamp(0,1)

@torch.no_grad()
def gray3_from_gray1(x1):
    """x1: [B,1,H,W] -> [B,3,H,W] for visualization"""
    return x1.repeat(1,3,1,1).clamp(0,1)

def make_example_panel(x_gray, y_lbls, logits, max_items=10, title=None, save_path=None):
    """
    Build a grid of triplets: [input gray | predicted color | ground truth color]
    """
    B = min(x_gray.size(0), max_items)
    pred_rgb = colourize_with_centroids(logits[:B])
    gt_rgb   = gt_rgb_from_labels(y_lbls[:B])
    gray3    = gray3_from_gray1(x_gray[:B])

    # Stack triplets along width
    triplets = torch.cat([gray3, pred_rgb, gt_rgb], dim=3)  # concat in width
    grid = vutils.make_grid(triplets, nrow=1)               # one sample per row

    if save_path is not None:
        vutils.save_image(grid, save_path)

    caption = title or "Left: input gray | Middle: predicted | Right: ground truth"
    return grid, caption

def train_one_epoch(model, loader, optimizer):
    model.train()
    total, count = 0.0, 0
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)              # [B,NC,H,W]
        loss   = F.cross_entropy(logits, y)  # per-pixel CE
        loss.backward()
        optimizer.step()

        bs = x.size(0)
        total += loss.item() * bs
        count += bs
    return total / count

@torch.no_grad()
def evaluate(model, loader, log_examples=False, step_tag="val"):
    model.eval()
    total, count = 0.0, 0
    last_batch = None

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        loss   = F.cross_entropy(logits, y)

        bs = x.size(0)
        total += loss.item() * bs
        count += bs
        last_batch = (x, y, logits)

    metrics = {"loss": total / count}

    if log_examples and last_batch is not None:
        x, y, logits = last_batch
        grid, cap = make_example_panel(x, y, logits, max_items=10, title=f"{step_tag} examples")
        # Log to W&B
        wandb.log({f"{step_tag}/examples": wandb.Image(grid, caption=cap)})
    return metrics

In [7]:
# %% [markdown]
# Train for at least 25 epochs, save & log the best checkpoint (lowest val loss),
# and log example colourizations + loss curves to W&B.

# %%
@dataclass
class TrainConfig:
    epochs: int = 25
    batch_size: int = 64
    nf: int = 16
    kernel_size: int = 3          # {3,5}
    lr: float = 3e-4
    weight_decay: float = 0.0
    optimizer: str = "adam"       # "sgd" or "adam"
    momentum: float = 0.9         # for SGD
    num_workers: int = 2

def train_run(cfg: TrainConfig):
    run = wandb.init(project=PROJECT, entity=ENTITY, config=cfg.__dict__, reinit=True)
    cfg = wandb.config  # W&B's FrozenDict

    # Data
    train_loader = make_loader(train_set, cfg.batch_size, True,  cfg.num_workers)
    val_loader   = make_loader(val_set,   cfg.batch_size, False, cfg.num_workers)

    # Model/opt
    model = ColorizationNet(NIC=NIC, NF=cfg.nf, NC=NC, kernel_size=cfg.kernel_size).to(device)
    opt   = make_optimizer(model, cfg.optimizer, lr=cfg.lr, wd=cfg.weight_decay, momentum=cfg.momentum)

    # Schedulers are optional; omitted per spec.
    best_val = float("inf")
    ckpt_dir = pathlib.Path("./checkpoints"); ckpt_dir.mkdir(exist_ok=True, parents=True)
    best_path = ckpt_dir / f"best_{run.name}.pt"

    history = {"train": [], "val": []}

    for epoch in range(1, cfg.epochs + 1):
        t0 = time.time()
        tr = train_one_epoch(model, train_loader, opt)
        val_metrics = evaluate(model, val_loader, log_examples=(epoch==1 or epoch%5==0), step_tag="val")

        history["train"].append(tr)
        history["val"].append(val_metrics["loss"])

        wandb.log({"epoch": epoch, "train/loss": tr, "val/loss": val_metrics["loss"]})

        if val_metrics["loss"] < best_val:
            best_val = val_metrics["loss"]
            torch.save({"model": model.state_dict(),
                        "cfg": dict(cfg),
                        "best_val": best_val,
                        "epoch": epoch}, best_path)
            wandb.run.summary["best_val_loss"] = float(best_val)
            wandb.save(str(best_path))
        print(f"[{epoch:03d}/{cfg.epochs}] train={tr:.4f} | val={val_metrics['loss']:.4f} | "
              f"time={time.time()-t0:.1f}s")

    # Plot curves locally and log
    plt.figure(figsize=(5,4))
    plt.plot(history["train"], label="train")
    plt.plot(history["val"],   label="val")
    plt.xlabel("epoch"); plt.ylabel("loss"); plt.title("Loss curves"); plt.legend()
    img_path = f"./loss_curves_{run.name}.png"
    plt.tight_layout(); plt.savefig(img_path, dpi=150); plt.close()
    wandb.log({"curves/loss": wandb.Image(img_path)})

    # Final: log 10 examples from validation
    evaluate(model, val_loader, log_examples=True, step_tag="final_val")

    run.finish()
    print("Best checkpoint:", best_path, "| best val loss:", best_val)
    return str(best_path), float(best_val)

# Example quick run (you can change nf, bs, lr etc., but keep epochs>=25 as required):
    # best_ckpt, best_val = train_run(TrainConfig(epochs=25, batch_size=64, nf=16, kernel_size=3, lr=3e-4, optimizer="adam"))


In [8]:
# %% [markdown]
# Evaluation on the test set + save a panel of 10 examples (input gray, predicted, ground truth).

# %%
@torch.no_grad()
def load_model_from_ckpt(ckpt_path):
    state = torch.load(ckpt_path, map_location="cpu")
    cfg = state.get("cfg", {})
    model = ColorizationNet(NIC=NIC, NF=cfg.get("nf",16), NC=NC, kernel_size=cfg.get("kernel_size",3))
    model.load_state_dict(state["model"])
    model.to(device).eval()
    return model, cfg

def export_test_examples(ckpt_path, out_path="test_examples.png", max_items=10):
    model, _cfg = load_model_from_ckpt(ckpt_path)
    loader = make_loader(test_set, batch_size=max_items, shuffle=False, num_workers=2)
    x, y = next(iter(loader))
    x = x.to(device); y = y.to(device)
    logits = model(x)
    grid, cap = make_example_panel(x, y, logits, max_items=max_items, title="Test examples", save_path=out_path)
    print("Saved:", out_path, "|", cap)
    return out_path

# Usage after training:
# export_test_examples(best_ckpt)


In [9]:
# %% [markdown]
# Hyperparameter Tuning with W&B Sweeps:
#  - LR: {1e-4, 3e-4, 1e-3, 3e-3}
#  - Batch size: {32, 64, 128}
#  - NF: {8, 16, 32}
#  - Kernel size: {3, 5} (note: padding=k//2 handled in model)
#  - Optimizer: {SGD, Adam}
# Launch 20 runs.

# %%
sweep_config = {
    "name": "q2-colourization-sweep",
    "method": "random",
    "metric": {"name": "val/loss", "goal": "minimize"},
    "parameters": {
        "epochs": {"value": 25},
        "batch_size": {"values": [32, 64, 128]},
        "nf": {"values": [8, 16, 32]},
        "kernel_size": {"values": [3, 5]},
        "lr": {"values": [1e-4, 3e-4, 1e-3, 3e-3]},
        "optimizer": {"values": ["sgd", "adam"]},
        "weight_decay": {"values": [0.0, 1e-4]},
        "momentum": {"value": 0.9},
        "num_workers": {"value": 2},
    }
}

def sweep_train():
    # W&B will pass config; reuse train_run internals but adapt to sweep
    run = wandb.init(project=PROJECT, entity=ENTITY, reinit=True)
    cfg = TrainConfig(**run.config.as_dict())
    best_ckpt, best_val = train_run(cfg)
    # attach best path for convenience
    run.summary["best_ckpt_path"] = best_ckpt
    run.finish()

# To start the sweep:
# sweep_id = wandb.sweep(sweep_config, project=PROJECT, entity=ENTITY)
# wandb.agent(sweep_id, function=sweep_train, count=20)


In [10]:
# %% [markdown]
# Utility: query W&B for the best run by validation loss and print its URL.

# %%
def report_best_run(entity=ENTITY, project=PROJECT):
    api = wandb.Api()
    runs = api.runs(f"{entity}/{project}")
    best = None
    for r in runs:
        val = r.summary.get("best_val_loss", None)
        if val is None:
            continue
        if (best is None) or (val < best.summary["best_val_loss"]):
            best = r
    if best is None:
        print("No completed runs with 'best_val_loss' found.")
        return None
    print("Best run:", best.name)
    print("Best val loss:", best.summary["best_val_loss"])
    print("URL:", best.url)
    print("Config:", best.config)
    return best

# Example after sweep:
# report_best_run()


In [11]:
# %% [markdown]
# If you didn't use W&B for images/curves, this cell shows how to export them directly after `train_run`.

# %%
def save_curves(history_json_path, out_path="loss_curves.png"):
    with open(history_json_path, "r") as f:
        H = json.load(f)
    plt.figure(figsize=(5,4))
    plt.plot(H["train"], label="train")
    plt.plot(H["val"],   label="val")
    plt.xlabel("epoch"); plt.ylabel("loss"); plt.title("Loss curves"); plt.legend()
    plt.tight_layout(); plt.savefig(out_path, dpi=150); plt.close()
    print("Saved curves to", out_path)

# Note: history is already logged to W&B in Cell 7; this optional cell is here
# only if you want an additional local copy from a saved JSON you maintain.

# %% [markdown]
# Gallery saver: create 10×10 panels of **(gray | predicted | ground-truth)** across a split,
# display them inline, and save each panel to disk. Works with your existing cells.

# %%
from PIL import Image
from IPython.display import display

@torch.no_grad()
def save_split_galleries(
    ckpt_path,
    split: str = "test",     # {"train","val","test"}
    rows: int = 10,
    cols: int = 10,
    out_prefix: str | None = None,
    display_inline: bool = True,
):
    """
    Loads the checkpoint, runs inference over the requested split, and for every
    chunk of rows*cols images creates a single big panel where each item is a
    triplet concatenated widthwise: [gray | predicted | ground truth].

    Saves:  out_prefix_p01.png, out_prefix_p02.png, ...
    Logs to W&B when available.
    Returns: list of saved file paths (strings).
    """
    assert rows > 0 and cols > 0, "rows/cols must be positive"
    per_panel = rows * cols

    # Load model
    model, _cfg = load_model_from_ckpt(ckpt_path)
    model.eval()

    # Use global centroids if present; else reload
    C = CENTROIDS if "CENTROIDS" in globals() else load_centroids(CENTROIDS_PATH).to(device)

    # Pick dataset for the split (use existing globals if defined)
    ds_map = {"train": globals().get("train_set"),
              "val":   globals().get("val_set"),
              "test":  globals().get("test_set")}
    ds = ds_map.get(split)

    if ds is None:
        # Fallback rebuild (keeps behavior consistent even if called in a fresh kernel)
        base_train = Cifar10Colorize(DATA_DIR, train=True,  centroids=C, precompute=True)
        base_test  = Cifar10Colorize(DATA_DIR, train=False, centroids=C, precompute=True)
        if split == "test":
            ds = base_test
        elif split == "val":
            VAL_SIZE = 5000
            TRAIN_SIZE = len(base_train) - VAL_SIZE
            ds = random_split(
                base_train, [TRAIN_SIZE, VAL_SIZE],
                generator=torch.Generator().manual_seed(SEED)
            )[1]
        else:
            ds = base_train

    loader = DataLoader(
        ds, batch_size=per_panel, shuffle=False,
        num_workers=2, pin_memory=(device.type == "cuda")
    )

    if out_prefix is None:
        out_prefix = f"gallery_{split}"

    saved_paths = []
    page = 1
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)

        # Build triplets: [B,3,H,3W] = concat along width: gray | pred | gt
        pred = colourize_with_centroids(logits, C)
        gt   = gt_rgb_from_labels(y, C)
        g3   = gray3_from_gray1(x)
        triplets = torch.cat([g3, pred, gt], dim=3)

        # Arrange into grid with `cols` items per row (rows * cols per panel)
        grid = vutils.make_grid(triplets, nrow=cols)

        # Save panel
        path = f"{out_prefix}_p{page:02d}.png"
        vutils.save_image(grid, path)
        saved_paths.append(path)
        print(f"✅ Saved: {path}")

        # Log to W&B (best-effort)
        try:
            wandb.log({f"{split}/gallery_p{page:02d}": wandb.Image(path)})
        except Exception:
            pass

        # Inline display (optional)
        if display_inline:
            display(Image.open(path))

        page += 1

    return saved_paths

# ---- Example usage (runs on your last/best checkpoint for the TEST split) ----
# This will generate 10×10=100 triplets per panel until the whole split is covered,
# displaying them inline and saving gallery_test_p01.png, p02, ...
# paths = save_split_galleries(best_ckpt_path, split="test", rows=10, cols=10, out_prefix="gallery_test")



In [12]:
import os, pathlib, time
import wandb
import matplotlib.pyplot as plt

# Override any previous ENTITY to avoid 404s
ENTITY = None  # use your logged-in user by default

def _wandb_safe_init(project, config, entity=None):
    """Init W&B; if it fails, fall back to offline mode."""
    try:
        init_kwargs = dict(project=project, config=config,
                           settings=wandb.Settings())
        if entity:  # only pass entity if provided
            init_kwargs["entity"] = entity
        run = wandb.init(**init_kwargs)
        return run
    except Exception as e:
        print("⚠️ W&B init failed → switching to offline mode.\nReason:", e)
        os.environ["WANDB_MODE"] = "offline"
        return wandb.init(project=project, config=config, mode="offline")

def train_run(cfg: TrainConfig):
    run = _wandb_safe_init(PROJECT, cfg.__dict__, entity=ENTITY)
    cfg = wandb.config  # use W&B's config object

    # Data
    train_loader = make_loader(train_set, cfg.batch_size, True,  2)
    val_loader   = make_loader(val_set,   cfg.batch_size, False, 2)

    # Model + Optimizer
    model = ColorizationNet(NIC=NIC, NF=cfg.nf, NC=NC, kernel_size=cfg.kernel_size).to(device)
    opt   = make_optimizer(model, cfg.optimizer, lr=cfg.lr,
                           wd=cfg.weight_decay, momentum=cfg.momentum)

    # Training
    best_val = float("inf")
    history = {"train": [], "val": []}
    ckpt_dir = pathlib.Path("./checkpoints"); ckpt_dir.mkdir(exist_ok=True, parents=True)
    best_path = ckpt_dir / f"best_{run.name}.pt"

    for epoch in range(1, cfg.epochs + 1):
        tr = train_one_epoch(model, train_loader, opt)
        val = evaluate(model, val_loader,
                       log_examples=(epoch == 1 or epoch % 5 == 0),
                       step_tag="val")
        history["train"].append(tr)
        history["val"].append(val["loss"])
        wandb.log({"epoch": epoch, "train/loss": tr, "val/loss": val["loss"]})

        if val["loss"] < best_val:
            best_val = val["loss"]
            torch.save({
                "model": model.state_dict(),
                "cfg": dict(cfg),
                "best_val": float(best_val),
                "epoch": epoch
            }, best_path)
            # Summary + best checkpoint artifact (best-effort)
            wandb.run.summary["best_val_loss"] = float(best_val)
            try:
                wandb.save(str(best_path))
            except Exception:
                pass

        print(f"[{epoch:03d}/{cfg.epochs}] train={tr:.4f} | val={val['loss']:.4f}")

    # Loss curves (also logged to W&B)
    plt.figure(figsize=(5,4))
    plt.plot(history["train"], label="train")
    plt.plot(history["val"],   label="val")
    plt.xlabel("epoch"); plt.ylabel("loss"); plt.title("Loss curves"); plt.legend()
    img_path = f"./loss_curves_{run.name}.png"
    plt.tight_layout(); plt.savefig(img_path, dpi=150); plt.close()
    try:
        wandb.log({"curves/loss": wandb.Image(img_path)})
    except Exception:
        pass

    # Final example panel from validation
    evaluate(model, val_loader, log_examples=True, step_tag="final_val")
    run.finish()
    print("Best checkpoint:", best_path, "| best val loss:", best_val)
    return str(best_path), float(best_val)

# --------- RUN IT ---------
run_cfg = TrainConfig(
    epochs=25, batch_size=64, nf=16, kernel_size=3,
    lr=3e-4, optimizer="adam", weight_decay=0.0, momentum=0.9, num_workers=2
)

best_ckpt_path, best_val = train_run(run_cfg)
print(f"\n✅ Training complete. Best checkpoint: {best_ckpt_path}\nBest val loss: {best_val:.6f}")

# Export 10 test examples (gray | predicted | ground truth)
panel_path = export_test_examples(best_ckpt_path, out_path="test_examples.png", max_items=10)
print(f"🖼️  Saved test examples panel at: {panel_path}")

[001/25] train=2.5157 | val=2.1459
[002/25] train=2.0659 | val=1.9912
[003/25] train=1.9731 | val=1.9191
[004/25] train=1.9321 | val=1.8886
[005/25] train=1.9015 | val=1.8620
[006/25] train=1.8848 | val=1.8461
[007/25] train=1.8707 | val=1.8367
[008/25] train=1.8612 | val=1.8352
[009/25] train=1.8501 | val=1.8128
[010/25] train=1.8387 | val=1.8061
[011/25] train=1.8305 | val=1.7921
[012/25] train=1.8229 | val=1.7891
[013/25] train=1.8190 | val=1.7794
[014/25] train=1.8156 | val=1.7704
[015/25] train=1.8088 | val=1.7649
[016/25] train=1.8044 | val=1.7649
[017/25] train=1.8006 | val=1.7600
[018/25] train=1.7914 | val=1.7631
[019/25] train=1.7918 | val=1.7738
[020/25] train=1.7884 | val=1.7461
[021/25] train=1.7867 | val=1.7520
[022/25] train=1.7821 | val=1.7525
[023/25] train=1.7789 | val=1.7445
[024/25] train=1.7737 | val=1.7414
[025/25] train=1.7737 | val=1.7392


epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
train/loss,█▄▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/loss,█▅▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▂▁▁▁▁▁▁
best_val_loss,1.73924
epoch,25
train/loss,1.77366
val/loss,1.73924


Best checkpoint: checkpoints/best_dutiful-fire-3.pt | best val loss: 1.7392391681671142

✅ Training complete. Best checkpoint: checkpoints/best_dutiful-fire-3.pt
Best val loss: 1.739239
Saved: test_examples.png | Test examples
🖼️  Saved test examples panel at: test_examples.png


In [13]:
import os, time, pathlib, random
import wandb
import matplotlib.pyplot as plt

# Fallbacks if not defined earlier
PROJECT = globals().get("PROJECT", "smai-q2-colorization")
ENTITY  = globals().get("ENTITY", None)  # don't force an entity

def _wandb_safe_init(project, config, entity=None):
    """Init W&B; if it fails, fall back to offline mode."""
    try:
        kw = dict(project=project, config=config,
                  settings=wandb.Settings())
        if entity: kw["entity"] = entity
        return wandb.init(**kw)
    except Exception as e:
        print("⚠️ W&B init failed → offline mode.\nReason:", e)
        os.environ["WANDB_MODE"] = "offline"
        return wandb.init(project=project, config=config, mode="offline")

def _normalize_cfg(cfg=None, **kwargs):
    """Merge defaults from TrainConfig (if present) with dict/kwargs."""
    base = {}
    if "TrainConfig" in globals():
        # pull defaults from dataclass if available
        base = {k: v for k, v in TrainConfig().__dict__.items()}
    # Allow cfg as dataclass or dict
    if cfg is not None:
        if "TrainConfig" in globals() and isinstance(cfg, TrainConfig):
            base.update(cfg.__dict__)
        elif isinstance(cfg, dict):
            base.update(cfg)
        else:
            raise TypeError("cfg must be TrainConfig, dict, or None")
    # Overlay kwargs
    base.update(kwargs)
    # Hard defaults if no dataclass existed
    base.setdefault("epochs", 25)
    base.setdefault("batch_size", 64)
    base.setdefault("nf", 16)
    base.setdefault("kernel_size", 3)
    base.setdefault("lr", 3e-4)
    base.setdefault("optimizer", "adam")
    base.setdefault("weight_decay", 0.0)
    base.setdefault("momentum", 0.9)
    base.setdefault("num_workers", 2)
    return base

def train_run(cfg=None, **kwargs):
    """
    Train once with given hyperparams.
    Usage examples:
      train_run(epochs=25, batch_size=64, nf=16, kernel_size=3, lr=3e-4, optimizer="adam")
      train_run({"epochs":25, "batch_size":32, ...})
      train_run(TrainConfig(epochs=25, ...))
    """
    conf = _normalize_cfg(cfg, **kwargs)
    run  = _wandb_safe_init(PROJECT, conf, entity=ENTITY)

    # ---------- Data ----------
    bs = conf["batch_size"]
    nw = conf["num_workers"]
    train_loader = make_loader(train_set, bs, True,  nw)
    val_loader   = make_loader(val_set,   bs, False, nw)

    # ---------- Model / Opt ----------
    model = ColorizationNet(NIC=NIC, NF=conf["nf"], NC=NC, kernel_size=conf["kernel_size"]).to(device)
    opt   = make_optimizer(model, conf["optimizer"], lr=conf["lr"],
                           wd=conf["weight_decay"], momentum=conf["momentum"])

    # ---------- Train loop ----------
    best_val = float("inf")
    history = {"train": [], "val": []}
    ckpt_dir = pathlib.Path("./checkpoints"); ckpt_dir.mkdir(exist_ok=True, parents=True)
    best_path = ckpt_dir / f"best_{run.name}.pt"

    for epoch in range(1, conf["epochs"] + 1):
        t0 = time.time()
        tr = train_one_epoch(model, train_loader, opt)
        val_metrics = evaluate(model, val_loader,
                               log_examples=(epoch == 1 or epoch % 5 == 0),
                               step_tag="val")
        history["train"].append(tr)
        history["val"].append(val_metrics["loss"])

        wandb.log({"epoch": epoch, "train/loss": tr, "val/loss": val_metrics["loss"]})
        if val_metrics["loss"] < best_val:
            best_val = val_metrics["loss"]
            torch.save({
                "model": model.state_dict(),
                "cfg": conf,
                "best_val": float(best_val),
                "epoch": epoch
            }, best_path)
            wandb.run.summary["best_val_loss"] = float(best_val)
            try: wandb.save(str(best_path))
            except: pass

        print(f"[{epoch:03d}/{conf['epochs']}] train={tr:.4f} | val={val_metrics['loss']:.4f} | "
              f"time={time.time()-t0:.1f}s")

    # Curves
    plt.figure(figsize=(5,4))
    plt.plot(history["train"], label="train")
    plt.plot(history["val"],   label="val")
    plt.xlabel("epoch"); plt.ylabel("loss"); plt.title("Loss curves"); plt.legend()
    img_path = f"./loss_curves_{run.name}.png"
    plt.tight_layout(); plt.savefig(img_path, dpi=150); plt.close()
    try: wandb.log({"curves/loss": wandb.Image(img_path)})
    except: pass

    # Final examples from val
    evaluate(model, val_loader, log_examples=True, step_tag="final_val")
    run.finish()

    print("Best checkpoint:", best_path, "| best val loss:", best_val)
    return str(best_path), float(best_val)

## 📝 Q2: Analytical Questions - Model Architecture Analysis

The model architecture is an **encoder-decoder** network for image colourization on CIFAR-10. The computation below is done symbolically in terms of $NIC$ (Input Channels, typically 1), $NF$ (Base Filters, e.g., 16), $NC$ (Output Classes/Channels, 24), and $k$ (Conv2d kernel size, e.g., 3 or 5).

**Assumptions for Spatial Calculations:**
1.  **Conv2d:** `padding = k // 2` and `stride = 1` (Maintains spatial size before pooling).
2.  **MaxPool2d:** `kernel_size = 2` and `stride = 2` (Halves spatial size).
3.  **ConvTranspose2d:** `kernel_size = 2` and `stride = 2` (Doubles spatial size).
4.  **$1\times1$ Classifier:** `kernel_size = 1` and `stride = 1` (Maintains spatial size).
5.  **Ignored Parameters:** Biases and Batch Normalization parameters are ignored for weight count.

| Layer Type | Input Shape $[C_{in}, H_{in}, W_{in}]$ | Output Shape $[C_{out}, H_{out}, W_{out}]$ | Weights ($\mathcal{W}_L$) |
| :---: | :---: | :---: | :---: |
| **Enc 1** (Conv2d + Pool) | $[NIC, S, S]$ | $[NF, S/2, S/2]$ | $NF \cdot NIC \cdot k^2$ |
| **Enc 2** (Conv2d + Pool) | $[NF, S/2, S/2]$ | $[2NF, S/4, S/4]$ | $2NF^2 \cdot k^2$ |
| **Enc 3** (Conv2d + Pool) | $[2NF, S/4, S/4]$ | $[4NF, S/8, S/8]$ | $8NF^2 \cdot k^2$ |
| **Dec 1** (ConvTranspose2d) | $[4NF, S/8, S/8]$ | $[2NF, S/4, S/4]$ | $2NF \cdot 4NF \cdot 2^2 = 32NF^2$ |
| **Dec 2** (ConvTranspose2d) | $[2NF, S/4, S/4]$ | $[NF, S/2, S/2]$ | $NF \cdot 2NF \cdot 2^2 = 8NF^2$ |
| **Dec 3** (ConvTranspose2d) | $[NF, S/2, S/2]$ | $[NC, S, S]$ | $NC \cdot NF \cdot 2^2 = 4NC \cdot NF$ |
| **Clas.** ($1\times1$ Conv2d) | $[NC, S, S]$ | $[NC, S, S]$ | $NC \cdot NC \cdot 1^2 = NC^2$ |

---

### I. Input Spatial Size: $S=32$ (CIFAR-10)

| Metric | Symbolic Formula |
| :--- | :--- |
| **1. Number of Weights ($\mathcal{W}$)** (Sum of $\mathcal{W}_L$) | $$\mathcal{W} = NF \cdot NIC \cdot k^2 + NF^2 (10 k^2 + 40) + 4 \cdot NC \cdot NF + NC^2$$ **(Correction:** Dec 1 weights are $4NF \cdot 2NF \cdot 2^2 = 32NF^2$. Dec 2 weights are $2NF \cdot NF \cdot 2^2 = 8NF^2$. Dec 3 weights are $NF \cdot NC \cdot 2^2 = 4NF \cdot NC$. The initial analysis was slightly off on the Decoder ConvTranspose weights (which are $C_{in} \cdot C_{out} \cdot k^2$ for Conv2d, but for `ConvTranspose2d` with $k=2, s=2$, the effective formula is simpler if you treat it as a linear layer on $4\times4$ inputs: $C_{in} \cdot C_{out} \cdot k^2$). Let's use the standard formula for $k=2, s=2$ deconv kernel weights: $C_{in} \cdot C_{out} \cdot k^2$.
* Dec 1: $4NF \cdot 2NF \cdot 2^2 = 32NF^2$
* Dec 2: $2NF \cdot NF \cdot 2^2 = 8NF^2$
* Dec 3: $NF \cdot NC \cdot 2^2 = 4NC \cdot NF$
* Total $NF^2$ terms: $2k^2 + 8k^2 + 32 + 8 = 10k^2 + 40$.
* Total $\mathcal{W} = NF \cdot NIC \cdot k^2 + NF^2 (10 k^2 + 40) + 4 \cdot NC \cdot NF + NC^2$ )

| Layer Type | $C_{in} \to C_{out}$ | H, W | Weights $\mathcal{W}_L$ |
| :---: | :---: | :---: | :---: |
| Enc 1 | $NIC \to NF$ | 32 | $NF \cdot NIC \cdot k^2$ |
| Enc 2 | $NF \to 2NF$ | 16 | $2NF^2 \cdot k^2$ |
| Enc 3 | $2NF \to 4NF$ | 8 | $8NF^2 \cdot k^2$ |
| Dec 1 | $4NF \to 2NF$ | 4 | $32NF^2$ |
| Dec 2 | $2NF \to NF$ | 8 | $8NF^2$ |
| Dec 3 | $NF \to NC$ | 16 | $4NC \cdot NF$ |
| Clas. | $NC \to NC$ | 32 | $NC^2$ |
| **Total** | | | $\mathbf{NF \cdot NIC \cdot k^2 + NF^2 (10 k^2 + 40) + 4 \cdot NC \cdot NF + NC^2}$ |

| Metric | Symbolic Formula |
| :--- | :--- |
| **2. Number of Outputs ($\mathcal{O}$)** (Sum of elements after ReLU/Clas.)| $$\mathcal{O} = NF \cdot (256+128+64+128+256) + NC \cdot (1024+1024)$$ |
| | $$\mathcal{O} = \mathbf{832 \cdot NF + 2048 \cdot NC}$$ |
| **3. Number of Connections ($\mathcal{C}$)** | $$\mathcal{C} = \sum_{L} (\mathcal{W}_L \times H_{out}^2 \text{ before pooling/transpose})$$ |
| | $$\mathcal{C} = \mathbf{1024 \cdot NF \cdot NIC \cdot k^2 + NF^2 (1024 \cdot k^2 + 4096) + 1024 \cdot NC \cdot NF + 1024 \cdot NC^2}$$ |

---

### II. Input Spatial Size: $S=64$

The input is $[B, NIC, 64, 64]$. All spatial dimensions are doubled, and the total area is quadrupled ($64^2 = 4096 = 4 \times 32^2$).

| Metric | Symbolic Formula | Comparison to $S=32$ |
| :--- | :--- | :--- |
| **1. Number of Weights ($\mathcal{W}'$**)| $$\mathcal{W}' = NF \cdot NIC \cdot k^2 + NF^2 (10 k^2 + 40) + 4 \cdot NC \cdot NF + NC^2$$ | $\mathbf{\mathcal{W}' = \mathcal{W}}$ (No Change) |
| **2. Number of Outputs ($\mathcal{O}'$**)| $$\mathcal{O}' = NF \cdot (1024+512+256+512+1024) + NC \cdot (4096+4096)$$|$$\mathbf{\mathcal{O}' = 4 \cdot \mathcal{O}}$$ |
| | $$\mathcal{O}' = \mathbf{3328 \cdot NF + 8192 \cdot NC}$$ | |
| **3. Number of Connections ($\mathcal{C}'$**)| $\mathcal{C}' = 4 \times \mathcal{C}$ (Connections scale by $4$) | $$\mathbf{\mathcal{C}' = 4 \cdot \mathcal{C}}$$ |
| | $\mathcal{C}' = \mathbf{4096 \cdot NF \cdot NIC \cdot k^2 + NF^2 (4096 \cdot k^2 + 16384) + 4096 \cdot NC \cdot NF + 4096 \cdot NC^2}$ | |

In [15]:
# %% [markdown]
# 🚀 Run 20 configurations (deterministic sample) across the required hyperparameter sets and save a CSV.

# %%
from itertools import product
import pandas as pd

SPACE = {
    "lr":         [1e-4, 3e-4, 1e-3, 3e-3],
    "batch_size": [32, 64, 128],
    "nf":         [8, 16, 32],
    "kernel_size":[3, 5],
    "optimizer":  ["sgd", "adam"],
}

# Build all combos, then pick a deterministic sample of 20 using the global SEED
all_combos = list(product(SPACE["lr"], SPACE["batch_size"], SPACE["nf"], SPACE["kernel_size"], SPACE["optimizer"]))
rng = random.Random(SEED if "SEED" in globals() else 1234)
selected = rng.sample(all_combos, 20)

results = []
for i, (lr, bs, nf, ks, opt) in enumerate(selected, 1):
    print(f"\n===== Run {i:02d}/20 | lr={lr} bs={bs} nf={nf} k={ks} opt={opt} =====")
    ckpt_path, best_val = train_run(
        epochs=25,
        batch_size=bs,
        nf=nf,
        kernel_size=ks,
        lr=lr,
        optimizer=opt,
        weight_decay=0.0,
        momentum=0.9,
        num_workers=2,
    )

    # Optional: export a 10-sample test panel for each run (comment out if heavy)
    export_test_examples(ckpt_path, out_path=f"test_examples_run{i:02d}.png", max_items=10)

    results.append({
        "run": i,
        "lr": lr,
        "batch_size": bs,
        "nf": nf,
        "kernel_size": ks,
        "optimizer": opt,
        "best_val_loss": best_val,
        "checkpoint": ckpt_path,
    })
    try: torch.cuda.empty_cache()
    except: pass

# Save & display table
df = pd.DataFrame(results).sort_values("best_val_loss").reset_index(drop=True)
display(df)
df.to_csv("sweep_results.csv", index=False)
print("💾 Saved: sweep_results.csv")

# Local best
best = df.iloc[0]
print("\n🏆 Best local run:")
for k in ["lr","batch_size","nf","kernel_size","optimizer","best_val_loss","checkpoint"]:
    print(f"  {k}: {best[k]}")

# If online, also print best W&B run URL (best-effort)
try:
    _ = report_best_run()
except Exception as e:
    print("W&B API not available (likely offline). Skipping online best report.\n", e)



===== Run 01/20 | lr=0.001 bs=32 nf=32 k=5 opt=adam =====


epoch,▁▂▃▄▅▅▆▇█
train/loss,█▇▆▅▄▃▂▂▁
val/loss,█▇▆▄▄▃▂▂▁
best_val_loss,2.8025
epoch,9
train/loss,2.82099
val/loss,2.8025


[001/25] train=1.9614 | val=1.7196 | time=9.6s
[002/25] train=1.6942 | val=1.5929 | time=9.7s
[003/25] train=1.6378 | val=1.5847 | time=10.8s
[004/25] train=1.6095 | val=1.5618 | time=10.0s
[005/25] train=1.5822 | val=1.5756 | time=9.8s
[006/25] train=1.5684 | val=1.5212 | time=9.3s
[007/25] train=1.5365 | val=1.5991 | time=10.1s
[008/25] train=1.5365 | val=1.5201 | time=10.0s
[009/25] train=1.5102 | val=1.4848 | time=10.0s
[010/25] train=1.4909 | val=1.4764 | time=9.8s
[011/25] train=1.4725 | val=1.4970 | time=10.2s
[012/25] train=1.4619 | val=1.4467 | time=10.1s
[013/25] train=1.4491 | val=1.4351 | time=10.2s
[014/25] train=1.4381 | val=1.4069 | time=9.6s
[015/25] train=1.4309 | val=1.4275 | time=9.8s
[016/25] train=1.4225 | val=1.3992 | time=10.2s
[017/25] train=1.4185 | val=1.5639 | time=10.8s
[018/25] train=1.4128 | val=1.3953 | time=10.1s
[019/25] train=1.4076 | val=1.5797 | time=9.3s
[020/25] train=1.4012 | val=1.4141 | time=12.4s
[021/25] train=1.3980 | val=1.3914 | time=10.0s


epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
train/loss,█▅▄▄▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
val/loss,█▆▅▅▅▄▆▄▃▃▄▃▂▂▂▂▅▂▅▂▂▂▁▂▁
best_val_loss,1.3588
epoch,25
train/loss,1.38555
val/loss,1.3588


Best checkpoint: checkpoints/best_fearless-valley-6.pt | best val loss: 1.3587970832824707
Saved: test_examples_run01.png | Test examples

===== Run 02/20 | lr=0.0003 bs=128 nf=16 k=5 opt=sgd =====


...

[001/25] train=2.6463 | val=2.2806 | time=8.5s
[002/25] train=2.1965 | val=2.0948 | time=9.4s
[003/25] train=2.0868 | val=2.0187 | time=9.3s
[004/25] train=2.0342 | val=1.9693 | time=8.9s
[005/25] train=2.0014 | val=1.9381 | time=8.8s
[006/25] train=1.9813 | val=1.9257 | time=9.2s
[007/25] train=1.9617 | val=1.9006 | time=9.2s
[008/25] train=1.9475 | val=1.8967 | time=8.4s
[009/25] train=1.9396 | val=1.8840 | time=9.2s
[010/25] train=1.9267 | val=1.8724 | time=9.1s
[011/25] train=1.9182 | val=1.8562 | time=8.3s
[012/25] train=1.9114 | val=1.8466 | time=9.2s
[013/25] train=1.8986 | val=1.8362 | time=9.2s
[014/25] train=1.8942 | val=1.8392 | time=8.7s
[015/25] train=1.8878 | val=1.8262 | time=8.9s
[016/25] train=1.8827 | val=1.8162 | time=9.2s
[017/25] train=1.8750 | val=1.8106 | time=9.1s
[018/25] train=1.8714 | val=1.8205 | time=8.6s
[019/25] train=1.8665 | val=1.8047 | time=9.2s
[020/25] train=1.8592 | val=1.8029 | time=9.3s
[021/25] train=1.8583 | val=1.8033 | time=8.3s
[022/25] trai

epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
train/loss,█▄▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
val/loss,█▅▄▄▃▃▃▃▂▂▂▂▂▂▂▁▁▂▁▁▁▁▁▁▁
best_val_loss,1.78287
epoch,25
train/loss,1.84429
val/loss,1.79446


Best checkpoint: checkpoints/best_lyric-yogurt-25.pt | best val loss: 1.7828679077148437
Saved: test_examples_run20.png | Test examples


,run,lr,batch_size,nf,kernel_size,optimizer,best_val_loss,checkpoint
0,1,0.0010,32,32,5,adam,1.358797,checkpoints/best_fearless-valley-6.pt
1,15,0.0010,64,32,5,adam,1.393525,checkpoints/best_summer-dream-20.pt
2,18,0.0030,128,32,5,adam,1.401794,checkpoints/best_gentle-firefly-23.pt
3,12,0.0030,64,32,5,adam,1.405350,checkpoints/best_elated-dream-17.pt
4,13,0.0010,64,16,5,adam,1.501638,checkpoints/best_cosmic-thunder-18.pt
5,7,0.0030,128,32,5,sgd,1.569924,checkpoints/best_helpful-butterfly-12.pt
6,14,0.0003,32,32,3,adam,1.611055,checkpoints/best_clean-armadillo-19.pt
7,16,0.0001,128,32,3,adam,1.697497,checkpoints/best_woven-pyramid-21.pt
8,4,0.0010,32,16,3,adam,1.714211,checkpoints/best_firm-gorge-9.pt
9,17,0.0001,32,8,5,adam,1.759105,checkpoints/best_trim-wildflower-22.pt


💾 Saved: sweep_results.csv

🏆 Best local run:
  lr: 0.001
  batch_size: 32
  nf: 32
  kernel_size: 5
  optimizer: adam
  best_val_loss: 1.3587970832824707
  checkpoint: checkpoints/best_fearless-valley-6.pt
W&B API not available (likely offline). Skipping online best report.
 Could not find project smai-q2-colorization
